### 전처리

In [1]:
# ============================================================
# 라이브러리 Import
# ============================================================

# 데이터 처리 및 분석
import os
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 통계 분석
from scipy import stats
from scipy.stats import shapiro, levene, ttest_ind, chi2_contingency, f_oneway
from scipy.stats import mannwhitneyu, fisher_exact, kruskal
from scipy.stats import skew, kurtosis
from statsmodels.stats.multicomp import pairwise_tukeyhsd, MultiComparison
import pingouin as pg
import scikit_posthocs as sp

# 머신러닝
from sklearn.preprocessing import MinMaxScaler

# 출력 설정
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 참고: seed 고정으로 팀원 간 동일한 결과 재현 가능
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


# 1. 데이터 로드

- metadata.csv 파일을 불러오는 첫 단계
- 총 7,565행, 34개 배터리 데이터  
- charge (충전), discharge (방전), impedance (내부저항)

In [2]:
import pandas as pd
import os

# 경로 설정
base_path = "data"  # 프로젝트 기준
meta_path = os.path.join(base_path, "metadata.csv")

# metadata 불러오기
df = pd.read_csv(meta_path)

# 확인
print(df.shape)
df.head()

(7565, 10)


,type,start_time,ambient_temperature,battery_id,test_id,uid,filename,Capacity,Re,Rct
0,discharge,[2010. 7. 21. 15. 0. ...,4,B0047,0,1,00001.csv,1.6743047446975208,NaN,NaN
1,impedance,[2010. 7. 21. 16. 53. ...,24,B0047,1,2,00002.csv,NaN,0.05605783343888099,0.20097016584458333
2,charge,[2010. 7. 21. 17. 25. ...,4,B0047,2,3,00003.csv,NaN,NaN,NaN
3,impedance,[2010 7 21 20 31 5],24,B0047,3,4,00004.csv,NaN,0.05319185850921101,0.16473399914864734
4,discharge,[2.0100e+03 7.0000e+00 2.1000e+01 2.1000e+01 2...,4,B0047,4,5,00005.csv,1.5243662105099023,NaN,NaN


# 2. start_time 전처리 

In [3]:
import pandas as pd

def clear_time(x):
    # 결측치 처리
    if pd.isna(x):
        return pd.NaT
    
    # 문자열로 변환
    x = str(x)

    nums = []
    current = ''

    # 숫자 관련 문자만 추출
    for ch in x:
        if ch.isdigit() or ch in ['.', 'e', 'E', '+', '-']:
            current += ch
        else:
            if current != '':
                nums.append(current)
                current = ''

    if current != '':
        nums.append(current)

    # 연월일시분초 최소 6개 필요
    if len(nums) < 6:
        return pd.NaT

    try:
        nums = list(map(float, nums[:6]))
    except:
        return pd.NaT

    year, month, day, hour, minute, second = nums

    # 범위 체크
    if not (2000 <= year <= 2100 and
            1 <= month <= 12 and
            1 <= day <= 31 and
            0 <= hour < 24 and
            0 <= minute < 60 and
            0 <= second < 60):
        return pd.NaT

    try:
        return pd.Timestamp(
            int(year), int(month), int(day),
            int(hour), int(minute), int(second)
        )
    except:
        return pd.NaT


df['start_time'] = df['start_time'].apply(clear_time)
df = df.dropna(subset=['start_time'])

# 3. 그룹 컬럼 추가

- 배터리 34개를 README 기준으로 A~I 9개 그룹으로 묶음.
- 그룹마다 온도·전류·종료 방식이 다르기 때문에 나눠서 관리
- group: 실험 조건이 같은 배터리 묶음A, B, C ... I
- end_reason: 실험이 왜 끝났는가EOL(수명 다함), crashed(크래시), censored(사이클 부족)
- analysis_role: 이 그룹을 어디에 쓸지 main(핵심), comparison(비교), anomaly(이상탐지), excluded(제외)


In [4]:
# 그룹 매핑 (A~I)
group_map = {
    'B0005':'A','B0006':'A','B0007':'A','B0018':'A',
    'B0025':'B','B0026':'B','B0027':'B','B0028':'B',
    'B0029':'C','B0030':'C','B0031':'C','B0032':'C',
    'B0033':'D','B0034':'D','B0036':'D',
    'B0038':'E','B0039':'E','B0040':'E',
    'B0041':'F','B0042':'F','B0043':'F','B0044':'F',
    'B0045':'G','B0046':'G','B0047':'G','B0048':'G',
    'B0049':'H','B0050':'H','B0051':'H','B0052':'H',
    'B0053':'I','B0054':'I','B0055':'I','B0056':'I',
}
 
# 종료 사유 매핑
# EOL      : 정상 수명 종료
# censored : EOL 미달 (사이클 부족)
# crashed  : SW 크래시로 실험 강제 중단
# QA_issue : 데이터 품질 문제 (조건 변경, 초기값 비정상)
end_reason_map = {
    'A': 'EOL',
    'B': 'censored',
    'C': 'censored',
    'D': 'QA_issue',
    'E': 'QA_issue',
    'F': 'QA_issue',
    'G': 'QA_issue',
    'H': 'crashed',
    'I': 'QA_issue',
}
 
# 그룹별 실험 조건 (프로토콜 매트릭스)
group_info = {
    'A': {'온도':'실온',    '방전전류':'2A CC',      'cutoff':'2.7/2.5/2.2/2.5V', '신뢰도':'정상'},
    'B': {'온도':'24°C',   '방전전류':'4A 스퀘어파', 'cutoff':'2.0~2.7V',          '신뢰도':'주의'},
    'C': {'온도':'43°C',   '방전전류':'4A',          'cutoff':'2.0~2.7V',          '신뢰도':'주의'},
    'D': {'온도':'24°C',   '방전전류':'4A/2A 혼합',  'cutoff':'2.0~2.7V',          '신뢰도':'비정상'},
    'E': {'온도':'24&44°C','방전전류':'1/2/4A 복합', 'cutoff':'2.2~2.7V',          '신뢰도':'비정상'},
    'F': {'온도':'4°C',    '방전전류':'4A/1A 혼합',  'cutoff':'2.0~2.7V',          '신뢰도':'비정상'},
    'G': {'온도':'4°C',    '방전전류':'1A',          'cutoff':'2.0~2.7V',          '신뢰도':'비정상'},
    'H': {'온도':'4°C',    '방전전류':'2A',          'cutoff':'2.0~2.7V',          '신뢰도':'심각'},
    'I': {'온도':'4°C',    '방전전류':'2A',          'cutoff':'2.0~2.7V',          '신뢰도':'비정상'},
}
 
# 분석 역할 매핑
analysis_role_map = {
    'A': 'main',        # 핵심 분석
    'B': 'excluded',    # 사이클 부족 제외
    'C': 'comparison',  # 온도 비교용
    'D': 'excluded',    # 초기값 비정상 제외
    'E': 'excluded',    # 조건 변경 복잡 제외
    'F': 'anomaly',     # 이상탐지 활용
    'G': 'anomaly',
    'H': 'anomaly',
    'I': 'anomaly',
}
 
df['group']        = df['battery_id'].map(group_map)
df['end_reason']   = df['group'].map(end_reason_map)
df['analysis_role']= df['group'].map(analysis_role_map)
 
print("\n그룹별 배터리 수 및 역할:")
summary = df.groupby(['group','end_reason','analysis_role'])['battery_id'].nunique().reset_index()
summary.columns = ['group','end_reason','analysis_role','배터리수']
print(summary.to_string(index=False))
 


그룹별 배터리 수 및 역할:
group end_reason analysis_role  배터리수
    A        EOL          main     4
    B   censored      excluded     4
    C   censored    comparison     4
    D   QA_issue      excluded     3
    E   QA_issue      excluded     3
    F   QA_issue       anomaly     4
    G   QA_issue       anomaly     4
    H    crashed       anomaly     4
    I   QA_issue       anomaly     4


원본 데이터는 충전·방전·임피던스가 한 테이블에 섞여 있음.
각 타입마다 의미 있는 컬럼이 다르기 때문에 반드시 분리

In [5]:
df_charge    = df[df['type'] == 'charge'].copy()
df_discharge = df[df['type'] == 'discharge'].copy()
df_impedance = df[df['type'] == 'impedance'].copy()
 
print("\n" + "=" * 55)
print("타입별 분리 완료")
print("=" * 55)
print(f"df_charge    : {len(df_charge):,}행")
print(f"df_discharge : {len(df_discharge):,}행")
print(f"df_impedance : {len(df_impedance):,}행")


타입별 분리 완료
df_charge    : 2,815행
df_discharge : 2,794행
df_impedance : 1,956행


# 4. Capacity 상태 flag 추가

Capacity 값을 5가지 상태로 분류

# flag 의미처리 방법

- valid: 정상값 (0.3 ~ 2.1Ah)분석에 사용
- zero: 정확히 0이상값으로 처리
- missing: NaN (빈 값)결측치로 처리
- low_anomaly0: 초과 0.3Ah 미만이상값 후보 (조건 변경 가능성 검토)
- impossible_high: 2.1Ah 초과 물리적으로 불가능한 값

In [6]:
df_discharge['Capacity'] = pd.to_numeric(
    df_discharge['Capacity'], errors='coerce'
)
 
def classify_capacity(cap):
    if pd.isna(cap):
        return 'missing'
    elif cap == 0:
        return 'zero'
    elif cap > 2.1:
        return 'impossible_high'
    elif cap < 0.3:
        return 'low_anomaly'
    else:
        return 'valid'
 
df_discharge['cap_flag'] = df_discharge['Capacity'].apply(classify_capacity)
 
print("\n" + "=" * 55)
print("Capacity flag 분포 (전체)")
print("=" * 55)
print(df_discharge['cap_flag'].value_counts())
 
print("\n그룹별 flag 분포:")
flag_grp = (
    df_discharge.groupby(['group','cap_flag'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
print(flag_grp.to_string(index=False))


Capacity flag 분포 (전체)
cap_flag
valid              2553
low_anomaly         193
missing              25
zero                 19
impossible_high       4
Name: count, dtype: int64

그룹별 flag 분포:
group  impossible_high  low_anomaly  missing  valid  zero
    A                0            0        0    636     0
    B                0            0        0    112     0
    C                0            0        0    160     0
    D                1            6        0    584     0
    E                0            2        0    139     0
    F                0          180        0    220     3
    G                0            0        0    277    11
    H                3            5       25     64     3
    I                0            0        0    361     2


In [7]:
# 기존 group_map 아래에 이것만 추가하면 됩니당

ambient_profile_map = {
    'A': '24°C_stable',
    'B': '24°C_stable',
    'C': '43°C_stable',
    'D': '24°C_stable',
    'E': '24_44°C_mixed',
    'F': '4°C_stable',
    'G': '4°C_stable',
    'H': '4°C_stable',
    'I': '4°C_stable',
}

load_profile_map = {
    'A': '2A_CC',
    'B': '4A_squarewave',
    'C': '4A_CC',
    'D': '2A_4A_mixed',
    'E': '1A_2A_4A_mixed',
    'F': '4A_1A_mixed',
    'G': '1A_CC',
    'H': '2A_CC',
    'I': '2A_CC',
}

cutoff_voltage_map = {
    'A': 2.7,
    'B': 2.7,
    'C': 2.7,
    'D': 2.5,
    'E': 2.5,
    'F': 2.7,
    'G': 2.7,
    'H': 2.7,
    'I': 2.7,
}

eol_rule_source_map = {
    'A': 'NASA_30%fade',
    'B': 'censored',
    'C': 'censored',
    'D': 'NASA_20%fade',
    'E': 'NASA_20%fade',
    'F': 'NASA_30%fade',
    'G': 'NASA_30%fade',
    'H': 'crashed',
    'I': 'NASA_30%fade',
}

# 기존 df['group'] = ... 아래에 추가하시면 됩니다
df['ambient_profile']  = df['group'].map(ambient_profile_map)
df['load_profile']     = df['group'].map(load_profile_map)
df['cutoff_voltage']   = df['group'].map(cutoff_voltage_map)
df['eol_rule_source']  = df['group'].map(eol_rule_source_map)

# 5. discharge 사이클 순번 추가

각 배터리마다 방전을 몇 번째 했는지 순번을 붙임.

- B0005의 첫 번째 방전 → discharge_cycle_no = 1
- B0005의 두 번째 방전 → discharge_cycle_no = 2
- B0006의 첫 번째 방전 → discharge_cycle_no = 1

In [8]:
df_discharge = df_discharge.sort_values(['battery_id','test_id'])
df_discharge['discharge_cycle_no'] = (
    df_discharge.groupby('battery_id').cumcount() + 1
)

## 6. 초기 Capacity 계산

각 배터리의 처음 상태(기준값)을 구함.
SOH 계산에 꼭 필요!

- 첫 번째 사이클 값 단독 사용 금지
- B0041처럼 초반에 실험 조건이 달라서 값이 비정상으로 낮은 경우가 있음.그래서 valid한 값 중 앞 5개의 중앙값으로 계산.

In [9]:
init_cap = (
    df_discharge[df_discharge['cap_flag'] == 'valid']
    .groupby('battery_id')['Capacity']
    .apply(lambda x: x.head(5).median())
    .rename('init_cap')
)
 
df_discharge = df_discharge.merge(init_cap, on='battery_id', how='left')
 
print("\n" + "=" * 55)
print("초기 Capacity (valid 구간 첫 5개 중앙값)")
print("=" * 55)
init_summary = df_discharge.groupby(['battery_id','group'])['init_cap'].first().reset_index()
print(init_summary.round(4).to_string(index=False))
 


초기 Capacity (valid 구간 첫 5개 중앙값)
battery_id group  init_cap
     B0005     A    1.8353
     B0006     A    2.0133
     B0007     A    1.8807
     B0018     A    1.8396
     B0025     B    1.8471
     B0026     B    1.8143
     B0027     B    1.8142
     B0028     B    1.7976
     B0029     C    1.8158
     B0030     C    1.7518
     B0031     C    1.8044
     B0032     C    1.8655
     B0033     D    1.2529
     B0034     D    1.6207
     B0036     D    1.8011
     B0038     E    1.0613
     B0039     E    0.4711
     B0040     E    0.7796
     B0041     F    1.1195
     B0042     F    1.7282
     B0043     F    1.6815
     B0044     F    1.6534
     B0045     G    0.8852
     B0046     G    1.5031
     B0047     G    1.5081
     B0048     G    1.4989
     B0049     H    1.3644
     B0050     H    1.5518
     B0051     H    1.2039
     B0052     H    1.3611
     B0053     I    1.1306
     B0054     I    1.0960
     B0055     I    1.2573
     B0056     I    1.2974


# 7. SOH 계산

SOH(State of Health) = 배터리 건강도

새 배터리 = SOH 100%
사용할수록 SOH가 낮아짐
SOH 80% 미만 = 교체 권장 수준

In [10]:
df_discharge['SOH_nominal'] = np.where(  # 모든 배터리를 공통 기준(2.0Ah)으로 비교하기 위한 SOH
    df_discharge['cap_flag'] == 'valid',  # 유효한 Capacity만 사용
    (df_discharge['Capacity'] / 2.0 * 100).round(2),  # 정격용량 대비 현재 용량 비율(%)
    np.nan  # 무효값은 제외
)

df_discharge['SOH_relative'] = np.where(  # 각 배터리의 시작점 대비 얼마나 감소했는지 보기 위한 SOH
    df_discharge['cap_flag'] == 'valid',  # 유효한 Capacity만 사용
    (df_discharge['Capacity'] / df_discharge['init_cap'] * 100).round(2),  # 초기용량 대비 현재 용량 비율(%)
    np.nan  # 무효값은 제외
)

# 8.EOL 탐지 & RUL 계산

- EOL(End of Life) = 수명 종료 시점
- RUL(Remaining Useful Life) = 앞으로 몇 번 더 쓸 수 있는가

In [11]:
# eol_cycles = (
#     df_discharge[
#         (df_discharge['cap_flag'] == 'valid') &
#         (df_discharge['SOH_relative'] < 80)
#     ]
#     .groupby('battery_id')['discharge_cycle_no']
#     .min()
#     .rename('eol_cycle')
# )

# df_discharge = df_discharge.merge(eol_cycles, on='battery_id', how='left')

# df_discharge['RUL'] = np.where(
#     df_discharge['eol_cycle'].notna(),
#     (df_discharge['eol_cycle'] - df_discharge['discharge_cycle_no']).clip(lower=0),
#     np.nan
# )

# print("\n" + "=" * 55)
# print("전체 그룹 EOL & RUL 요약")
# print("=" * 55)
# eol_summary = (
#     df_discharge.groupby('battery_id')
#     .agg(
#         group        = ('group',               'first'),
#         analysis_role= ('analysis_role',       'first'),
#         end_reason   = ('end_reason',          'first'),
#         eol_cycle    = ('eol_cycle',           'max'),
#         total_cycles = ('discharge_cycle_no',  'max'),
#         rul_available= ('RUL',                 lambda x: x.notna().sum()),
#     )
#     .reset_index()
# )
# eol_summary['eol_달성'] = eol_summary['eol_cycle'].notna().map({True:'O', False:'X'})
# print(eol_summary.to_string(index=False))

In [12]:
# EOL 정의 (SOH_relative 기준으로 통일)
eol_cycles = (
    df_discharge[
        (df_discharge['cap_flag'] == 'valid') &
        (df_discharge['SOH_relative'] < 80)
    ]
    .groupby('battery_id')['discharge_cycle_no']
    .min()
    .rename('eol_cycle')
)

df_discharge = df_discharge.merge(eol_cycles, on='battery_id', how='left')

# RUL 계산
df_discharge['RUL'] = np.where(
    (df_discharge['eol_cycle'].notna()) &
    (df_discharge['discharge_cycle_no'] <= df_discharge['eol_cycle']),
    df_discharge['eol_cycle'] - df_discharge['discharge_cycle_no'],
    np.nan
)

print("\n" + "=" * 55)
print("전체 그룹 EOL & RUL 요약")
print("=" * 55)
eol_summary = (
    df_discharge.groupby('battery_id')
    .agg(
        group         = ('group', 'first'),
        analysis_role = ('analysis_role', 'first'),
        end_reason    = ('end_reason', 'first'),
        eol_cycle     = ('eol_cycle', 'max'),
        total_cycles  = ('discharge_cycle_no', 'max'),
        rul_available = ('RUL', lambda x: x.notna().any())
    )
    .reset_index()
)

eol_summary['eol_달성'] = eol_summary['eol_cycle'].notna().map({True:'O', False:'X'})
print(eol_summary.to_string(index=False))


전체 그룹 EOL & RUL 요약
battery_id group analysis_role end_reason  eol_cycle  total_cycles  rul_available eol_달성
     B0005     A          main        EOL      107.0           168           True      O
     B0006     A          main        EOL       61.0           168           True      O
     B0007     A          main        EOL      125.0           168           True      O
     B0018     A          main        EOL       78.0           132           True      O
     B0025     B      excluded   censored        NaN            28          False      X
     B0026     B      excluded   censored        6.0            28           True      O
     B0027     B      excluded   censored        NaN            28          False      X
     B0028     B      excluded   censored        NaN            28          False      X
     B0029     C    comparison   censored        NaN            40          False      X
     B0030     C    comparison   censored        NaN            40          False      X
 

# 9.그룹별 DataFrame 분리

- df_main: A / EDA·통계검증·ML 
- df_comparison:C / 온도별 비교 분석
- df_anomaly: F·G·H·I / 이상탐지 분석
- df_excluded: B·D·E /분석 제외

In [13]:
df_main       = df_discharge[df_discharge['analysis_role'] == 'main'].copy()
df_comparison = df_discharge[df_discharge['analysis_role'] == 'comparison'].copy()
df_anomaly    = df_discharge[df_discharge['analysis_role'] == 'anomaly'].copy()
df_excluded   = df_discharge[df_discharge['analysis_role'] == 'excluded'].copy()

# 10. impedance 전처리

- Re(전해질 저항)와 Rct(전하 전달 저항)를 숫자 타입으로 변환.

In [14]:
group_dfs = {}
for grp in sorted(df_discharge['group'].unique()):
    group_dfs[grp] = df_discharge[df_discharge['group'] == grp].copy()
 
print("\n" + "=" * 55)
print("역할별 DataFrame 분리 결과")
print("=" * 55)
print(f"df_main       (그룹A)    : {len(df_main):>4}행  {df_main['battery_id'].nunique()}개 배터리")
print(f"df_comparison (그룹C)    : {len(df_comparison):>4}행  {df_comparison['battery_id'].nunique()}개 배터리")
print(f"df_anomaly    (F·G·H·I) : {len(df_anomaly):>4}행  {df_anomaly['battery_id'].nunique()}개 배터리")
print(f"df_excluded   (B·D·E)   : {len(df_excluded):>4}행  {df_excluded['battery_id'].nunique()}개 배터리")


역할별 DataFrame 분리 결과
df_main       (그룹A)    :  636행  4개 배터리
df_comparison (그룹C)    :  160행  4개 배터리
df_anomaly    (F·G·H·I) : 1154행  16개 배터리
df_excluded   (B·D·E)   :  844행  10개 배터리


# 파생컬럼
- group: 실험 조건 그룹 (A~I)
- end_reason: 종료 사유
- analysis_role: 분석 역할 (main/comparison/anomaly/excluded)
- cap_flag: Capacity 상태 분류
- discharge_cycle_no: 배터리별 방전 순번
- init_cap: 초기 Capacity (첫 5개 중앙값)
- SOH: 배터리 건강도 (%)
- eol_cycle:EOL 도달 사이클 번호
- RUL:잔존 방전 사이클 수

# 테이블 생성

- df_discharge_processed.csv: 전체 34개 배터리 방전 데이터 (flag 포함)
- df_A_main.csv: 그룹A 메인 분석용
- df_C_comparison.csv: 그룹C 온도 비교용
- df_anomaly.csv: 그룹F·G·H·I 이상탐지용
- df_group_A.csv ~ df_group_I.csv: 그룹별 개별 파일필요할 때 꺼내 쓰기
- df_impedance_processed.csv: 전체 impedance (Re·Rct)
- df_imp_A_main.csv: 그룹A impedance
- df_imp_C_comparison.csv: 그룹C impedance
- df_ml_dataset.csvML: 학습용 데이터셋



In [15]:
df_impedance['Re']  = pd.to_numeric(df_impedance['Re'],  errors='coerce')
df_impedance['Rct'] = pd.to_numeric(df_impedance['Rct'], errors='coerce')
df_impedance['group']         = df_impedance['battery_id'].map(group_map)
df_impedance['end_reason']    = df_impedance['group'].map(end_reason_map)
df_impedance['analysis_role'] = df_impedance['group'].map(analysis_role_map)
df_impedance = df_impedance.sort_values(['battery_id','test_id'])
df_impedance['impedance_cycle_no'] = (
    df_impedance.groupby('battery_id').cumcount() + 1
)

# impedance도 역할별 분리
df_imp_main       = df_impedance[df_impedance['analysis_role'] == 'main'].copy()
df_imp_comparison = df_impedance[df_impedance['analysis_role'] == 'comparison'].copy()
df_imp_anomaly    = df_impedance[df_impedance['analysis_role'] == 'anomaly'].copy()

print("\nimpedance 전처리 완료:")
print(f"df_imp_main       : {len(df_imp_main):>4}행")
print(f"df_imp_comparison : {len(df_imp_comparison):>4}행")
print(f"df_imp_anomaly    : {len(df_imp_anomaly):>4}행")


impedance 전처리 완료:
df_imp_main       :  887행
df_imp_comparison :   68행
df_imp_anomaly    :  557행


In [16]:
# (같은 배터리의 impedance 사이클 번호 기준으로 매핑)
imp_A = df_imp_main[['battery_id','impedance_cycle_no','Re','Rct']].copy()
 
# discharge cycle과 impedance cycle을 battery_id 기준으로 nearest join
# 간단하게: impedance의 평균값을 discharge에 붙이는 방식
re_rct_mean = (
    df_imp_main.groupby('battery_id')[['Re','Rct']]
    .mean()
    .reset_index()
    .rename(columns={'Re':'Re_mean','Rct':'Rct_mean'})
)
 
df_ml = (
    df_main[df_main['cap_flag'] == 'valid']
    [['battery_id','group','discharge_cycle_no',
      'SOH','ambient_temperature','RUL']]
    .dropna(subset=['RUL'])
    .merge(re_rct_mean, on='battery_id', how='left')
    .copy()
)
 
print("\n" + "=" * 55)
print("ML용 데이터셋")
print("=" * 55)
print(f"총 행 수        : {len(df_ml)}")
print(f"배터리 수       : {df_ml['battery_id'].nunique()}")
print(f"결측치 여부     :\n{df_ml.isnull().sum()}")
print(f"\n샘플 (앞 5행):")
print(df_ml.head().to_string(index=False))

KeyError: "['SOH'] not in index"

In [ ]:
# os.makedirs(base_path, exist_ok=True)
 
# # 전체 discharge (모든 그룹, flag 포함)
# df_discharge.to_csv(
#     os.path.join(base_path, 'df_discharge_processed.csv'), index=False
# )
 
# # 역할별 저장
# df_main.to_csv(
#     os.path.join(base_path, 'df_A_main.csv'), index=False
# )
# df_comparison.to_csv(
#     os.path.join(base_path, 'df_C_comparison.csv'), index=False
# )
# df_anomaly.to_csv(
#     os.path.join(base_path, 'df_anomaly.csv'), index=False
# )
 
# # 그룹별 개별 저장
# for grp, gdf in group_dfs.items():
#     gdf.to_csv(
#         os.path.join(base_path, f'df_group_{grp}.csv'), index=False
#     )
 
# # impedance 저장
# df_impedance.to_csv(
#     os.path.join(base_path, 'df_impedance_processed.csv'), index=False
# )
# df_imp_main.to_csv(
#     os.path.join(base_path, 'df_imp_A_main.csv'), index=False
# )
# df_imp_comparison.to_csv(
#     os.path.join(base_path, 'df_imp_C_comparison.csv'), index=False
# )
 
# # ML 데이터셋
# df_ml.to_csv(
#     os.path.join(base_path, 'df_ml_dataset.csv'), index=False
# )
 
# print("\n" + "=" * 55)
# print("전체 저장 완료")
# print("=" * 55)
# print("df_discharge_processed.csv  — 전체 34개 배터리 discharge")
# print("df_A_main.csv               — 그룹A 메인 분석")
# print("df_C_comparison.csv         — 그룹C 온도 비교")
# print("df_anomaly.csv              — 그룹F·G·H·I 이상탐지")
# print("df_group_{A~I}.csv          — 그룹별 개별 파일")
# print("df_impedance_processed.csv  — 전체 impedance")
# print("df_imp_A_main.csv           — 그룹A impedance")
# print("df_imp_C_comparison.csv     — 그룹C impedance")
# print("df_ml_dataset.csv           — ML 학습용 데이터셋")
# print("\n다음 단계: 02_EDA.ipynb")

## rul_subset 생성
 
 현재 데이터상 RUl을 계산하기에는 main date가 제일 적합

main데이터에서만 rul계산을 위해 따로 뽑고 수식만들기

In [ ]:
rul_subset = eol_summary[eol_summary['analysis_role']=='main']

# RUL subset에 해당하는 battery만 필터
rul_ids = rul_subset['battery_id']

rul_df = df_discharge[df_discharge['battery_id'].isin(rul_ids)].copy()

# RUL 계산
rul_df['RUL'] = rul_df['eol_cycle'] - rul_df['discharge_cycle_no']

In [ ]:
rul_subset

,battery_id,group,analysis_role,end_reason,eol_cycle,total_cycles,rul_available,eol_달성
0,B0005,A,main,EOL,107.0,168,True,O
1,B0006,A,main,EOL,61.0,168,True,O
2,B0007,A,main,EOL,125.0,168,True,O
3,B0018,A,main,EOL,78.0,132,True,O


In [ ]:
# RUL subset에 해당하는 battery만 필터
rul_ids = rul_subset['battery_id']

rul_df = df_discharge[df_discharge['battery_id'].isin(rul_ids)].copy()

# RUL 계산
rul_df['RUL'] = rul_df['eol_cycle'] - rul_df['discharge_cycle_no']

In [ ]:
rul_df[rul_df['RUL'] == 0]

,type,start_time,ambient_temperature,battery_id,test_id,uid,filename,Capacity,Re,Rct,group,end_reason,analysis_role,cap_flag,discharge_cycle_no,init_cap,SOH_nominal,SOH_relative,eol_cycle,RUL
106,discharge,2008-05-13 09:24:04,24,B0005,378,5499,05499.csv,1.453901,NaN,NaN,A,EOL,main,valid,107,1.835349,72.70,79.22,107.0,0.0
228,discharge,2008-05-02 03:53:34,24,B0006,201,4706,04706.csv,1.608850,NaN,NaN,A,EOL,main,valid,61,2.013326,80.44,79.91,61.0,0.0
460,discharge,2008-05-17 17:15:37,24,B0007,448,6185,06185.csv,1.503196,NaN,NaN,A,EOL,main,valid,125,1.880663,75.16,79.93,125.0,0.0
581,discharge,2008-08-05 20:21:46,24,B0018,191,6544,06544.csv,1.468019,NaN,NaN,A,EOL,main,valid,78,1.839602,73.40,79.80,78.0,0.0
